In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import datetime as dt

from pathlib import Path
import os
os.chdir("./..")

from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import TimeSeriesSplit

pd.set_option("display.max_columns", None)

# Loading data

In [2]:
alarms = pd.read_csv("data/alarms/alarms_data_preprocessed_v2.csv")
weather = pd.read_csv("data/weather/weather_data_preprocessed_v3.csv")
isw = pd.read_csv("data/isw/isw_data_preprocessed_v5.csv")
telegram = pd.read_csv("data/telegram/telegram_hourly_features_v3.csv")

In [3]:
alarms.head(3)

,region_id,region_title,region_city,all_region,start,end,start_hour,start_minute
0,12,Львівська область,Львівська обл.,1,2022-02-24 07:43:17,2022-02-24 09:52:28,7,43
1,23,Чернігівська область,Чернігівська обл.,1,2022-02-24 14:00:43,2022-02-24 17:11:43,14,0
2,3,Вінницька область,Вінницька обл.,1,2022-02-24 15:40:42,2022-02-24 16:10:42,15,40


In [4]:
weather.head(3)

,temp,feelslike,humidity,dew,precip,precipprob,preciptype,windspeed,winddir,pressure,visibility,cloudcover,uvindex,conditions,real_hour_datetime,city
0,2.4,-1.6,89.18,0.8,0.0,0.0,['snow'],15.5,275.6,1020.0,0.0,91.5,0.0,Overcast,2022-02-24 00:00:00,Lutsk
1,2.4,-1.5,87.90,0.6,0.0,0.0,['snow'],14.8,280.3,1021.0,0.2,88.2,0.0,Partially cloudy,2022-02-24 01:00:00,Lutsk
2,2.9,-0.8,88.58,1.2,0.0,0.0,['snow'],14.4,310.0,1022.0,10.0,100.0,0.0,Overcast,2022-02-24 02:00:00,Lutsk


In [5]:
isw.head(3)

,text_length,isw_cluster_0,isw_cluster_1,isw_cluster_2,isw_cluster_3,isw_cluster_4,isw_cluster_5,isw_cluster_6,isw_cluster_7,isw_cluster_8,isw_cluster_9,anomaly_count_7d,avg_dist_centroid_7d,news_count_7d,topic_entropy_7d,topic_entropy_30d,date,news_velocity_30d,news_velocity_7d,centroid_shift_7d,avg_dist_centroid_30d,dom_cluster_share_7d,anomaly_count_30d,centroid_shift_30d,news_count_30d,dom_cluster_share_30d
0,13953,0,0,0,0,0,0,0,0,0,1,1,0.345617,7,0.598270,0.589003,2022-03-28,28,0,0.229950,0.396254,0.714286,3,0.565055,29,0.724138
1,10309,0,0,0,1,0,0,0,0,0,0,1,0.340311,7,0.410116,0.589003,2022-03-29,27,0,0.201195,0.393526,0.857143,3,0.527332,29,0.724138
2,30148,0,0,0,2,0,0,0,0,0,0,1,0.339411,7,0.598270,0.589003,2022-03-30,26,0,0.197885,0.387930,0.714286,3,0.512615,29,0.724138


In [6]:
telegram.head(3)

,datetime,messages_count,has_threat_sum,nlp_артобстрілу,nlp_бпла,nlp_відбій,nlp_відбій_тривоги,nlp_дніпропетровська,nlp_донецька,nlp_запорізька,nlp_нікополь,nlp_нікополь_нікопольська,nlp_нікопольська,nlp_повітряна,nlp_повітряна_тривога,nlp_тривога,nlp_тривоги,nlp_харківська,msg_count_last_3h,msg_count_last_24h,threat_diff_1h
0,2022-02-24 02:00:00+02:00,1.0,0.0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1.0,1.0,0.0
1,2022-02-24 03:00:00+02:00,0.0,0.0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1.0,1.0,0.0
2,2022-02-24 04:00:00+02:00,1.0,0.0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,2.0,2.0,0.0


# Preparing data for merging

All data have different granularity, so it need to be unified. I want to make granularity 1 hour, which would be best option for our task.

Key idea is to merge data by time and region id. Each row will have composed key created by `time` and `region_id` columns

## Alarms

In [7]:
alarms.head()

,region_id,region_title,region_city,all_region,start,end,start_hour,start_minute
0,12,Львівська область,Львівська обл.,1,2022-02-24 07:43:17,2022-02-24 09:52:28,7,43
1,23,Чернігівська область,Чернігівська обл.,1,2022-02-24 14:00:43,2022-02-24 17:11:43,14,0
2,3,Вінницька область,Вінницька обл.,1,2022-02-24 15:40:42,2022-02-24 16:10:42,15,40
3,19,Харківська область,Харківська обл.,1,2022-02-24 20:11:47,2022-02-24 20:59:47,20,11
4,18,Тернопільська область,Тернопільська обл.,1,2022-02-25 01:59:36,2022-02-25 09:00:19,1,59


In [8]:
alarms.info()

<class 'pandas.DataFrame'>
RangeIndex: 76147 entries, 0 to 76146
Data columns (total 8 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   region_id     76147 non-null  int64
 1   region_title  76147 non-null  str  
 2   region_city   76147 non-null  str  
 3   all_region    76147 non-null  int64
 4   start         76147 non-null  str  
 5   end           76147 non-null  str  
 6   start_hour    76147 non-null  int64
 7   start_minute  76147 non-null  int64
dtypes: int64(4), str(4)
memory usage: 12.1 MB


Convert `start`, `end` columns to datetime format

In [9]:
alarms["start"] = pd.to_datetime(alarms["start"])
alarms["end"] = pd.to_datetime(alarms["end"])

In [10]:
alarms

,region_id,region_title,region_city,all_region,start,end,start_hour,start_minute
0,12,Львівська область,Львівська обл.,1,2022-02-24 07:43:17,2022-02-24 09:52:28,7,43
1,23,Чернігівська область,Чернігівська обл.,1,2022-02-24 14:00:43,2022-02-24 17:11:43,14,0
2,3,Вінницька область,Вінницька обл.,1,2022-02-24 15:40:42,2022-02-24 16:10:42,15,40
3,19,Харківська область,Харківська обл.,1,2022-02-24 20:11:47,2022-02-24 20:59:47,20,11
4,18,Тернопільська область,Тернопільська обл.,1,2022-02-25 01:59:36,2022-02-25 09:00:19,1,59
...,...,...,...,...,...,...,...,...
76142,9,Київська область,Київська обл.,1,2026-03-16 22:28:50,2026-03-16 23:18:03,22,28
76143,9,Київська область,Київ,0,2026-03-16 22:34:21,2026-03-16 23:17:13,22,34
76144,17,Сумська область,Сумська обл.,1,2026-03-16 22:37:12,2026-03-17 07:26:30,22,37
76145,19,Харківська область,Харківська обл.,1,2026-03-16 23:27:16,2026-03-17 00:18:41,23,27


In [11]:
regions = alarms.groupby(['region_id', 'region_city']).size().reset_index().drop(columns=0)
regions

,region_id,region_city
0,1,Чернівецька обл.
1,2,Волинська обл.
2,3,Вінницька обл.
3,4,Дніпропетровська обл.
4,5,Донецька обл.
5,6,Житомирська обл.
6,7,Закарпатська обл.
7,8,Запорізька обл.
8,9,Київ
9,9,Київська обл.


In [12]:
alarms.isna().sum()

region_id       0
region_title    0
region_city     0
all_region      0
start           0
end             0
start_hour      0
start_minute    0
dtype: int64

In [13]:
alarms.start.min(), alarms.end.max()

(Timestamp('2022-02-24 07:43:17'), Timestamp('2026-03-17 07:26:30'))

Make 1 hour granularity

In [14]:
# create a helper column with the list of hours per alarm
from app.core.features.alarms_features import explode_by_hour, fix_regions

alarms = fix_regions(alarms)
alarms.head()

alarm_expanded = explode_by_hour(alarms)
alarm_expanded['time'] = pd.to_datetime(alarm_expanded['time'])
alarm_expanded.head()

,region_id,time,region,alarm,has_started,has_ended
0,3,2022-02-26 16:00:00,Хмельницька обл.,1,1,0
1,3,2022-02-26 17:00:00,Хмельницька обл.,1,0,1
2,3,2022-02-27 17:00:00,Хмельницька обл.,1,1,0
3,3,2022-02-27 18:00:00,Хмельницька обл.,1,0,1
4,3,2022-02-27 19:00:00,Хмельницька обл.,1,1,0


In [15]:
alarm_expanded.info()

<class 'pandas.DataFrame'>
RangeIndex: 179633 entries, 0 to 179632
Data columns (total 6 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   region_id    179633 non-null  int64         
 1   time         179633 non-null  datetime64[us]
 2   region       179633 non-null  str           
 3   alarm        179633 non-null  int64         
 4   has_started  179633 non-null  int64         
 5   has_ended    179633 non-null  int64         
dtypes: datetime64[us](1), int64(4), str(1)
memory usage: 13.0 MB


In [16]:
alarm_expanded.duplicated(subset=["region_id", "time"]).sum()

np.int64(0)

In [17]:
alarm_expanded = alarm_expanded.drop_duplicates(subset=["region_id", "time"])

In [18]:
alarm_expanded.shape

(179633, 6)

## Weather

In [19]:
weather.head()

,temp,feelslike,humidity,dew,precip,precipprob,preciptype,windspeed,winddir,pressure,visibility,cloudcover,uvindex,conditions,real_hour_datetime,city
0,2.4,-1.6,89.18,0.8,0.0,0.0,['snow'],15.5,275.6,1020.0,0.0,91.5,0.0,Overcast,2022-02-24 00:00:00,Lutsk
1,2.4,-1.5,87.90,0.6,0.0,0.0,['snow'],14.8,280.3,1021.0,0.2,88.2,0.0,Partially cloudy,2022-02-24 01:00:00,Lutsk
2,2.9,-0.8,88.58,1.2,0.0,0.0,['snow'],14.4,310.0,1022.0,10.0,100.0,0.0,Overcast,2022-02-24 02:00:00,Lutsk
3,2.3,-1.3,86.63,0.3,0.0,0.0,['snow'],13.3,295.1,1021.0,0.1,92.0,0.0,Overcast,2022-02-24 03:00:00,Lutsk
4,1.9,-1.8,87.85,0.1,0.0,0.0,['snow'],13.3,305.8,1021.0,0.0,93.8,0.0,Overcast,2022-02-24 04:00:00,Lutsk


In [20]:
weather.real_hour_datetime.min(), weather.real_hour_datetime.max()

('2022-02-24 00:00:00', '2026-03-16 23:00:00')

In [21]:
weather.isna().sum()

temp                       0
feelslike                  0
humidity                   0
dew                        0
precip                     0
precipprob                 0
preciptype            717975
windspeed                  0
winddir                    0
pressure                   0
visibility                 0
cloudcover                 0
uvindex                    0
conditions                 0
real_hour_datetime         0
city                       0
dtype: int64

In [22]:
weather.loc[weather.preciptype.isna()].sample(3)

,temp,feelslike,humidity,dew,precip,precipprob,preciptype,windspeed,winddir,pressure,visibility,cloudcover,uvindex,conditions,real_hour_datetime,city
529848,30.1,28.7,28.38,9.8,0.0,0.0,NaN,14.4,41.5,1018.0,10.0,97.7,4.0,Overcast,2024-08-31 15:00:00,Cherkasy
83005,19.9,19.9,69.69,14.2,0.0,0.0,NaN,0.0,65.4,1017.2,10.0,2.8,0.0,Clear,2022-07-23 00:00:00,Khmelnytskyi
429305,20.4,20.4,49.89,9.6,0.0,0.0,NaN,7.2,360.0,1013.8,10.0,40.0,0.0,Partially cloudy,2023-10-01 21:00:00,Mykolaiv


In [23]:
weather["preciptype"] = weather.preciptype.fillna("None")

In [24]:
weather.isna().sum()

temp                  0
feelslike             0
humidity              0
dew                   0
precip                0
precipprob            0
preciptype            0
windspeed             0
winddir               0
pressure              0
visibility            0
cloudcover            0
uvindex               0
conditions            0
real_hour_datetime    0
city                  0
dtype: int64

### Adding `region_id` column

In [25]:
weather_df = weather.copy()

In [26]:
from app.core.features.weather_features import add_region_ids

weather_df = add_region_ids(weather_df)

In [27]:
weather_df["time"] = pd.to_datetime(weather_df.pop("real_hour_datetime"))

In [28]:
weather_df

,temp,feelslike,humidity,dew,precip,precipprob,preciptype,windspeed,winddir,pressure,visibility,cloudcover,uvindex,conditions,city,region_id,time
0,2.4,-1.6,89.18,0.8,0.0,0.0,['snow'],15.5,275.6,1020.0,0.0,91.5,0.0,Overcast,Lutsk,8,2022-02-24 00:00:00
1,2.4,-1.5,87.90,0.6,0.0,0.0,['snow'],14.8,280.3,1021.0,0.2,88.2,0.0,Partially cloudy,Lutsk,8,2022-02-24 01:00:00
2,2.9,-0.8,88.58,1.2,0.0,0.0,['snow'],14.4,310.0,1022.0,10.0,100.0,0.0,Overcast,Lutsk,8,2022-02-24 02:00:00
3,2.3,-1.3,86.63,0.3,0.0,0.0,['snow'],13.3,295.1,1021.0,0.1,92.0,0.0,Overcast,Lutsk,8,2022-02-24 03:00:00
4,1.9,-1.8,87.85,0.1,0.0,0.0,['snow'],13.3,305.8,1021.0,0.0,93.8,0.0,Overcast,Lutsk,8,2022-02-24 04:00:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
844410,10.4,10.4,38.34,-3.2,0.0,0.0,None,8.6,148.6,1014.0,10.0,6.1,0.0,Clear,Lviv,27,2026-03-16 19:00:00
844411,9.3,8.1,42.51,-2.8,0.0,0.0,None,8.3,146.5,1016.0,10.0,46.7,0.0,Partially cloudy,Lviv,27,2026-03-16 20:00:00
844412,8.7,7.4,45.60,-2.4,0.0,0.0,None,8.3,157.4,1016.0,10.0,58.3,0.0,Partially cloudy,Lviv,27,2026-03-16 21:00:00
844413,8.2,6.7,48.23,-2.1,0.0,0.0,None,8.6,168.3,1016.0,10.0,58.1,0.0,Partially cloudy,Lviv,27,2026-03-16 22:00:00


## Telegram

In [29]:
telegram.head()

,datetime,messages_count,has_threat_sum,nlp_артобстрілу,nlp_бпла,nlp_відбій,nlp_відбій_тривоги,nlp_дніпропетровська,nlp_донецька,nlp_запорізька,nlp_нікополь,nlp_нікополь_нікопольська,nlp_нікопольська,nlp_повітряна,nlp_повітряна_тривога,nlp_тривога,nlp_тривоги,nlp_харківська,msg_count_last_3h,msg_count_last_24h,threat_diff_1h
0,2022-02-24 02:00:00+02:00,1.0,0.0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1.0,1.0,0.0
1,2022-02-24 03:00:00+02:00,0.0,0.0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1.0,1.0,0.0
2,2022-02-24 04:00:00+02:00,1.0,0.0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,2.0,2.0,0.0
3,2022-02-24 05:00:00+02:00,17.0,3.0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,18.0,19.0,3.0
4,2022-02-24 06:00:00+02:00,7.0,0.0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,25.0,26.0,-3.0


In [30]:
telegram.info()

<class 'pandas.DataFrame'>
RangeIndex: 35441 entries, 0 to 35440
Data columns (total 21 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   datetime                   35441 non-null  str    
 1   messages_count             35441 non-null  float64
 2   has_threat_sum             35441 non-null  float64
 3   nlp_артобстрілу            35441 non-null  int64  
 4   nlp_бпла                   35441 non-null  int64  
 5   nlp_відбій                 35441 non-null  int64  
 6   nlp_відбій_тривоги         35441 non-null  int64  
 7   nlp_дніпропетровська       35441 non-null  int64  
 8   nlp_донецька               35441 non-null  int64  
 9   nlp_запорізька             35441 non-null  int64  
 10  nlp_нікополь               35441 non-null  int64  
 11  nlp_нікополь_нікопольська  35441 non-null  int64  
 12  nlp_нікопольська           35441 non-null  int64  
 13  nlp_повітряна              35441 non-null  int64  
 14  n

In [31]:
tg_df = telegram.copy()
tg_df.rename({"datetime": "time"}, axis=1, inplace=True)
tg_df.time = pd.to_datetime(tg_df.time, utc=True) \
                .dt.tz_convert("Europe/Kyiv") \
                .dt.floor("h", ambiguous="infer")

In [32]:
tg_df.time.dtype

datetime64[us, Europe/Kyiv]

In [33]:
tg_df.duplicated(subset="time").sum()

np.int64(0)

## ISW

In [34]:
isw.head()

,text_length,isw_cluster_0,isw_cluster_1,isw_cluster_2,isw_cluster_3,isw_cluster_4,isw_cluster_5,isw_cluster_6,isw_cluster_7,isw_cluster_8,isw_cluster_9,anomaly_count_7d,avg_dist_centroid_7d,news_count_7d,topic_entropy_7d,topic_entropy_30d,date,news_velocity_30d,news_velocity_7d,centroid_shift_7d,avg_dist_centroid_30d,dom_cluster_share_7d,anomaly_count_30d,centroid_shift_30d,news_count_30d,dom_cluster_share_30d
0,13953,0,0,0,0,0,0,0,0,0,1,1,0.345617,7,0.598270,0.589003,2022-03-28,28,0,0.229950,0.396254,0.714286,3,0.565055,29,0.724138
1,10309,0,0,0,1,0,0,0,0,0,0,1,0.340311,7,0.410116,0.589003,2022-03-29,27,0,0.201195,0.393526,0.857143,3,0.527332,29,0.724138
2,30148,0,0,0,2,0,0,0,0,0,0,1,0.339411,7,0.598270,0.589003,2022-03-30,26,0,0.197885,0.387930,0.714286,3,0.512615,29,0.724138
3,45026,0,0,0,1,0,0,0,0,0,2,1,0.273573,7,0.598270,0.619376,2022-03-31,25,0,0.193133,0.377509,0.714286,3,0.526493,29,0.689655
4,14288,0,0,0,1,0,0,0,0,0,0,1,0.295166,7,0.598270,0.589003,2022-04-01,24,0,0.183463,0.370606,0.714286,3,0.547304,29,0.724138


In [35]:
isw.info()

<class 'pandas.DataFrame'>
RangeIndex: 1227 entries, 0 to 1226
Data columns (total 26 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   text_length            1227 non-null   int64  
 1   isw_cluster_0          1227 non-null   int64  
 2   isw_cluster_1          1227 non-null   int64  
 3   isw_cluster_2          1227 non-null   int64  
 4   isw_cluster_3          1227 non-null   int64  
 5   isw_cluster_4          1227 non-null   int64  
 6   isw_cluster_5          1227 non-null   int64  
 7   isw_cluster_6          1227 non-null   int64  
 8   isw_cluster_7          1227 non-null   int64  
 9   isw_cluster_8          1227 non-null   int64  
 10  isw_cluster_9          1227 non-null   int64  
 11  anomaly_count_7d       1227 non-null   int64  
 12  avg_dist_centroid_7d   1227 non-null   float64
 13  news_count_7d          1227 non-null   int64  
 14  topic_entropy_7d       1227 non-null   float64
 15  topic_entropy_3

In [36]:
isw.date = pd.to_datetime(isw.date)

In [37]:
isw.date

0      2022-03-28
1      2022-03-29
2      2022-03-30
3      2022-03-31
4      2022-04-01
          ...    
1222   2026-03-06
1223   2026-03-07
1224   2026-03-08
1225   2026-03-09
1226   2026-03-10
Name: date, Length: 1227, dtype: datetime64[us]

In [38]:
isw.isna().sum()

text_length              0
isw_cluster_0            0
isw_cluster_1            0
isw_cluster_2            0
isw_cluster_3            0
isw_cluster_4            0
isw_cluster_5            0
isw_cluster_6            0
isw_cluster_7            0
isw_cluster_8            0
isw_cluster_9            0
anomaly_count_7d         0
avg_dist_centroid_7d     0
news_count_7d            0
topic_entropy_7d         0
topic_entropy_30d        0
date                     0
news_velocity_30d        0
news_velocity_7d         0
centroid_shift_7d        0
avg_dist_centroid_30d    0
dom_cluster_share_7d     0
anomaly_count_30d        0
centroid_shift_30d       0
news_count_30d           0
dom_cluster_share_30d    0
dtype: int64

In [39]:
isw.duplicated(subset="date").sum()

np.int64(0)

# Merging

## Creating spine

Idea is to create spine by multiplying hours by regions and then merge everythin to it by time and region_id

In [40]:
timeline = pd.date_range(
    alarms.start.min().floor("h"),
    alarms.end.max().ceil("h"),
    freq="h"
)

region_ids = alarms.region_id.unique()

expected_length = len(timeline) * len(region_ids)

print(f"Number of hours: {len(timeline)}")
print(f"Number of regions: {len(region_ids)}")
print()
print(f"Expected length of spine: {expected_length}")

Number of hours: 35570
Number of regions: 24

Expected length of spine: 853680


In [41]:
spine = pd.MultiIndex.from_product([region_ids, timeline], names=["region_id", "time"]) \
            .to_frame(index=False) \
            .sort_values(["region_id", "time"]) \
            .reset_index(drop=True)

In [42]:
spine.shape

(853680, 2)

In [43]:
spine.head()

,region_id,time
0,3,2022-02-24 07:00:00
1,3,2022-02-24 08:00:00
2,3,2022-02-24 09:00:00
3,3,2022-02-24 10:00:00
4,3,2022-02-24 11:00:00


## Merging alarms to spine

In [44]:
merged_df = spine.merge(alarm_expanded, on=["region_id", "time"], how="left")
merged_df["alarm"] = merged_df["alarm"].fillna(0).astype(int)

In [45]:
merged_df

,region_id,time,region,alarm,has_started,has_ended
0,3,2022-02-24 07:00:00,NaN,0,NaN,NaN
1,3,2022-02-24 08:00:00,NaN,0,NaN,NaN
2,3,2022-02-24 09:00:00,NaN,0,NaN,NaN
3,3,2022-02-24 10:00:00,NaN,0,NaN,NaN
4,3,2022-02-24 11:00:00,NaN,0,NaN,NaN
...,...,...,...,...,...,...
853675,31,2026-03-17 04:00:00,NaN,0,NaN,NaN
853676,31,2026-03-17 05:00:00,NaN,0,NaN,NaN
853677,31,2026-03-17 06:00:00,NaN,0,NaN,NaN
853678,31,2026-03-17 07:00:00,NaN,0,NaN,NaN


## Merging weather

In [46]:
merged_df = merged_df.merge(weather_df, on=["region_id", "time"], how="left")

In [47]:
merged_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 853680 entries, 0 to 853679
Data columns (total 21 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   region_id    853680 non-null  int64         
 1   time         853680 non-null  datetime64[us]
 2   region       179633 non-null  str           
 3   alarm        853680 non-null  int64         
 4   has_started  179633 non-null  float64       
 5   has_ended    179633 non-null  float64       
 6   temp         844247 non-null  float64       
 7   feelslike    844247 non-null  float64       
 8   humidity     844247 non-null  float64       
 9   dew          844247 non-null  float64       
 10  precip       844247 non-null  float64       
 11  precipprob   844247 non-null  float64       
 12  preciptype   844247 non-null  str           
 13  windspeed    844247 non-null  float64       
 14  winddir      844247 non-null  float64       
 15  pressure     844247 non-null  float64       


## Merging telegram

In [48]:
merged_df.time = merged_df.time.dt.tz_localize("Europe/Kyiv", ambiguous="NaT", nonexistent="NaT")

In [49]:
tg_df.shape

(35441, 21)

In [50]:
merged_df = merged_df.merge(tg_df, on=["time"], how="left")

In [51]:
merged_df.shape

(853680, 41)

## Merging ISW

In [52]:
isw.shape

(1227, 26)

In [53]:
merged_df["date"] = merged_df["time"].dt.date

In [54]:
merged_df.date = pd.to_datetime(merged_df.date)

In [55]:
merged_df = merged_df.merge(isw, on="date", how="left")

In [56]:
merged_df = merged_df.drop(columns="date")

In [57]:
merged_df.shape

(853680, 66)

## Result

In [58]:
merged_df.head()

,region_id,time,region,alarm,has_started,has_ended,temp,feelslike,humidity,dew,precip,precipprob,preciptype,windspeed,winddir,pressure,visibility,cloudcover,uvindex,conditions,city,messages_count,has_threat_sum,nlp_артобстрілу,nlp_бпла,nlp_відбій,nlp_відбій_тривоги,nlp_дніпропетровська,nlp_донецька,nlp_запорізька,nlp_нікополь,nlp_нікополь_нікопольська,nlp_нікопольська,nlp_повітряна,nlp_повітряна_тривога,nlp_тривога,nlp_тривоги,nlp_харківська,msg_count_last_3h,msg_count_last_24h,threat_diff_1h,text_length,isw_cluster_0,isw_cluster_1,isw_cluster_2,isw_cluster_3,isw_cluster_4,isw_cluster_5,isw_cluster_6,isw_cluster_7,isw_cluster_8,isw_cluster_9,anomaly_count_7d,avg_dist_centroid_7d,news_count_7d,topic_entropy_7d,topic_entropy_30d,news_velocity_30d,news_velocity_7d,centroid_shift_7d,avg_dist_centroid_30d,dom_cluster_share_7d,anomaly_count_30d,centroid_shift_30d,news_count_30d,dom_cluster_share_30d
0,3,2022-02-24 07:00:00+02:00,NaN,0,NaN,NaN,1.2,-1.1,93.72,0.3,0.0,0.0,None,7.2,287.2,1024.0,0.5,100.0,0.0,Overcast,Khmelnytskyi,21.0,2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,45.0,47.0,2.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,3,2022-02-24 08:00:00+02:00,NaN,0,NaN,NaN,1.4,-0.9,95.09,0.7,0.4,100.0,"['rain', 'snow']",7.2,280.0,1024.7,10.0,100.0,1.0,"Snow, Rain, Overcast",Khmelnytskyi,20.0,2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,48.0,67.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,3,2022-02-24 09:00:00+02:00,NaN,0,NaN,NaN,1.4,-1.4,91.05,0.1,0.0,0.0,None,9.0,295.3,1025.0,23.7,88.5,0.0,Partially cloudy,Khmelnytskyi,19.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,60.0,86.0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,3,2022-02-24 10:00:00+02:00,NaN,0,NaN,NaN,2.0,-0.6,85.35,-0.2,0.0,0.0,None,8.6,311.1,1026.0,23.5,88.7,1.0,Partially cloudy,Khmelnytskyi,13.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,52.0,99.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,3,2022-02-24 11:00:00+02:00,NaN,0,NaN,NaN,2.2,-0.9,96.49,1.7,0.0,0.0,None,10.8,320.0,1025.8,10.0,100.0,1.0,Overcast,Khmelnytskyi,22.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,54.0,121.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [59]:
merged_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 853680 entries, 0 to 853679
Data columns (total 66 columns):
 #   Column                     Non-Null Count   Dtype                      
---  ------                     --------------   -----                      
 0   region_id                  853680 non-null  int64                      
 1   time                       853488 non-null  datetime64[us, Europe/Kyiv]
 2   region                     179633 non-null  str                        
 3   alarm                      853680 non-null  int64                      
 4   has_started                179633 non-null  float64                    
 5   has_ended                  179633 non-null  float64                    
 6   temp                       844247 non-null  float64                    
 7   feelslike                  844247 non-null  float64                    
 8   humidity                   844247 non-null  float64                    
 9   dew                        844247 non-null  floa

In [60]:
merged_df.shape

(853680, 66)

In [61]:
print(f"Expected length: {expected_length}")
print(f"Actual length: {merged_df.shape[0]}")

Expected length: 853680
Actual length: 853680


In [62]:
merged_df.region_id.nunique()

24

In [63]:
merged_df.city.nunique()

23

In [64]:
merged_df.region.nunique()

24

# Preprocessing merged data

In [65]:
df = merged_df.copy()

In [66]:
df = df.loc[~df.time.isna()]

In [67]:
with pd.option_context("display.max_rows", None):
    print(df.isna().sum())

region_id                         0
time                              0
region                       673889
alarm                             0
has_started                  673889
has_ended                    673889
temp                           9336
feelslike                      9336
humidity                       9336
dew                            9336
precip                         9336
precipprob                     9336
preciptype                     9336
windspeed                      9336
winddir                        9336
pressure                       9336
visibility                     9336
cloudcover                     9336
uvindex                        9336
conditions                     9336
city                           9336
messages_count                 3216
has_threat_sum                 3216
nlp_артобстрілу                3216
nlp_бпла                       3216
nlp_відбій                     3216
nlp_відбій_тривоги             3216
nlp_дніпропетровська        

In [68]:
df = df.drop(["city"], axis=1)

In [69]:
df = df.loc[~df.temp.isna()]

df.text_length = df.text_length.fillna(-1)
df.preciptype = df.preciptype.fillna("None")

isw_cluster_cols = [col for col in df.columns if col.startswith("isw_cluster")]
df[isw_cluster_cols] = df[isw_cluster_cols].fillna(0)

df.visibility = df.visibility.ffill()
df.uvindex = df.uvindex.ffill()

df = df.loc[~df.messages_count.isna()]

df["year"] = df["time"].dt.year
df["month"] = df["time"].dt.month
df["day"] = df["time"].dt.day
df["hour"] = df["time"].dt.hour
df['hour_of_day'] = df['time'].dt.hour
df['day_of_week'] = df['time'].dt.dayofweek
df['is_weekend'] = (df['day_of_week'] >= 5).astype(int)

In [70]:
with pd.option_context("display.max_rows", None):
    print(df.isna().sum())

region_id                         0
time                              0
region                       669749
alarm                             0
has_started                  669749
has_ended                    669749
temp                              0
feelslike                         0
humidity                          0
dew                               0
precip                            0
precipprob                        0
preciptype                        0
windspeed                         0
winddir                           0
pressure                          0
visibility                        0
cloudcover                        0
uvindex                           0
conditions                        0
messages_count                    0
has_threat_sum                    0
nlp_артобстрілу                   0
nlp_бпла                          0
nlp_відбій                        0
nlp_відбій_тривоги                0
nlp_дніпропетровська              0
nlp_донецька                

In [71]:
df = df.ffill() # using forward fill to fill missng values with values from previous records

In [72]:
with pd.option_context("display.max_rows", None):
    print(df.isna().sum())

region_id                      0
time                           0
region                        57
alarm                          0
has_started                   57
has_ended                     57
temp                           0
feelslike                      0
humidity                       0
dew                            0
precip                         0
precipprob                     0
preciptype                     0
windspeed                      0
winddir                        0
pressure                       0
visibility                     0
cloudcover                     0
uvindex                        0
conditions                     0
messages_count                 0
has_threat_sum                 0
nlp_артобстрілу                0
nlp_бпла                       0
nlp_відбій                     0
nlp_відбій_тривоги             0
nlp_дніпропетровська           0
nlp_донецька                   0
nlp_запорізька                 0
nlp_нікополь                   0
nlp_нікопо

In [73]:
df.loc[df.isw_cluster_1.isna(), "time"].unique()

<DatetimeArray>
[]
Length: 0, dtype: datetime64[us, Europe/Kyiv]

missing values only at first month of war, so we can drop them.

In [74]:
df = df.dropna(axis=0)

In [75]:
df.isna().sum().sum()

np.int64(0)

In [76]:
df = df.reset_index(drop=True)

In [77]:
encoder = LabelEncoder()

cat_cols = list(df.select_dtypes(include=["str"]).columns)

for col_name in cat_cols:
    df[col_name] = encoder.fit_transform(df[col_name])

In [78]:
df.head(3)

,region_id,time,region,alarm,has_started,has_ended,temp,feelslike,humidity,dew,precip,precipprob,preciptype,windspeed,winddir,pressure,visibility,cloudcover,uvindex,conditions,messages_count,has_threat_sum,nlp_артобстрілу,nlp_бпла,nlp_відбій,nlp_відбій_тривоги,nlp_дніпропетровська,nlp_донецька,nlp_запорізька,nlp_нікополь,nlp_нікополь_нікопольська,nlp_нікопольська,nlp_повітряна,nlp_повітряна_тривога,nlp_тривога,nlp_тривоги,nlp_харківська,msg_count_last_3h,msg_count_last_24h,threat_diff_1h,text_length,isw_cluster_0,isw_cluster_1,isw_cluster_2,isw_cluster_3,isw_cluster_4,isw_cluster_5,isw_cluster_6,isw_cluster_7,isw_cluster_8,isw_cluster_9,anomaly_count_7d,avg_dist_centroid_7d,news_count_7d,topic_entropy_7d,topic_entropy_30d,news_velocity_30d,news_velocity_7d,centroid_shift_7d,avg_dist_centroid_30d,dom_cluster_share_7d,anomaly_count_30d,centroid_shift_30d,news_count_30d,dom_cluster_share_30d,year,month,day,hour,hour_of_day,day_of_week,is_weekend
0,3,2022-03-28 00:00:00+03:00,20,0,0.0,1.0,0.2,-3.3,58.88,-6.9,0.0,0.0,0,10.8,300.0,1029.3,10.0,0.0,0.0,0,13.0,5.0,0.0,0.0,4.0,4.0,0.0,0.0,0.0,0.0,0.0,0.0,5.0,5.0,5.0,4.0,0.0,86.0,289.0,-3.0,13953.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.345617,7.0,0.59827,0.589003,28.0,0.0,0.22995,0.396254,0.714286,3.0,0.565055,29.0,0.724138,2022,3,28,0,0,0,0
1,3,2022-03-28 01:00:00+03:00,20,0,0.0,1.0,-0.1,-1.8,48.02,-9.8,0.0,0.0,0,5.0,291.3,1030.0,24.1,1.6,0.0,0,8.0,2.0,0.0,0.0,3.0,3.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,4.0,0.0,45.0,285.0,-3.0,13953.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.345617,7.0,0.59827,0.589003,28.0,0.0,0.22995,0.396254,0.714286,3.0,0.565055,29.0,0.724138,2022,3,28,1,1,0,0
2,3,2022-03-28 02:00:00+03:00,20,0,0.0,1.0,-0.6,-0.6,51.39,-9.4,0.0,0.0,0,4.7,271.0,1030.0,24.1,2.5,0.0,0,7.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,28.0,269.0,-2.0,13953.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.345617,7.0,0.59827,0.589003,28.0,0.0,0.22995,0.396254,0.714286,3.0,0.565055,29.0,0.724138,2022,3,28,2,2,0,0


In [79]:
df.sort_values(by=["region_id", "time"])

,region_id,time,region,alarm,has_started,has_ended,temp,feelslike,humidity,dew,precip,precipprob,preciptype,windspeed,winddir,pressure,visibility,cloudcover,uvindex,conditions,messages_count,has_threat_sum,nlp_артобстрілу,nlp_бпла,nlp_відбій,nlp_відбій_тривоги,nlp_дніпропетровська,nlp_донецька,nlp_запорізька,nlp_нікополь,nlp_нікополь_нікопольська,nlp_нікопольська,nlp_повітряна,nlp_повітряна_тривога,nlp_тривога,nlp_тривоги,nlp_харківська,msg_count_last_3h,msg_count_last_24h,threat_diff_1h,text_length,isw_cluster_0,isw_cluster_1,isw_cluster_2,isw_cluster_3,isw_cluster_4,isw_cluster_5,isw_cluster_6,isw_cluster_7,isw_cluster_8,isw_cluster_9,anomaly_count_7d,avg_dist_centroid_7d,news_count_7d,topic_entropy_7d,topic_entropy_30d,news_velocity_30d,news_velocity_7d,centroid_shift_7d,avg_dist_centroid_30d,dom_cluster_share_7d,anomaly_count_30d,centroid_shift_30d,news_count_30d,dom_cluster_share_30d,year,month,day,hour,hour_of_day,day_of_week,is_weekend
0,3,2022-03-28 00:00:00+03:00,20,0,0.0,1.0,0.2,-3.3,58.88,-6.9,0.0,0.0,0,10.8,300.0,1029.3,10.0,0.0,0.0,0,13.0,5.0,0.0,0.0,4.0,4.0,0.0,0.0,0.0,0.0,0.0,0.0,5.0,5.0,5.0,4.0,0.0,86.0,289.0,-3.0,13953.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.345617,7.0,0.598270,0.589003,28.0,0.0,0.22995,0.396254,0.714286,3.0,0.565055,29.0,0.724138,2022,3,28,0,0,0,0
1,3,2022-03-28 01:00:00+03:00,20,0,0.0,1.0,-0.1,-1.8,48.02,-9.8,0.0,0.0,0,5.0,291.3,1030.0,24.1,1.6,0.0,0,8.0,2.0,0.0,0.0,3.0,3.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,4.0,0.0,45.0,285.0,-3.0,13953.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.345617,7.0,0.598270,0.589003,28.0,0.0,0.22995,0.396254,0.714286,3.0,0.565055,29.0,0.724138,2022,3,28,1,1,0,0
2,3,2022-03-28 02:00:00+03:00,20,0,0.0,1.0,-0.6,-0.6,51.39,-9.4,0.0,0.0,0,4.7,271.0,1030.0,24.1,2.5,0.0,0,7.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,28.0,269.0,-2.0,13953.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.345617,7.0,0.598270,0.589003,28.0,0.0,0.22995,0.396254,0.714286,3.0,0.565055,29.0,0.724138,2022,3,28,2,2,0,0
3,3,2022-03-28 03:00:00+03:00,20,0,0.0,1.0,-0.8,-4.5,59.99,-7.6,0.0,0.0,0,10.8,270.0,1029.4,10.0,40.0,0.0,4,11.0,10.0,0.0,0.0,0.0,0.0,2.0,0.0,2.0,0.0,0.0,0.0,9.0,9.0,9.0,0.0,2.0,26.0,275.0,10.0,13953.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.345617,7.0,0.598270,0.589003,28.0,0.0,0.22995,0.396254,0.714286,3.0,0.565055,29.0,0.724138,2022,3,28,3,3,0,0
4,3,2022-03-28 04:00:00+03:00,20,0,0.0,1.0,-1.3,-1.3,56.25,-8.9,0.0,0.0,0,4.3,228.8,1029.0,24.1,18.8,0.0,0,9.0,1.0,0.0,0.0,6.0,6.0,2.0,0.0,2.0,0.0,0.0,0.0,1.0,1.0,2.0,6.0,2.0,27.0,283.0,-9.0,13953.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.345617,7.0,0.598270,0.589003,28.0,0.0,0.22995,0.396254,0.714286,3.0,0.565055,29.0,0.724138,2022,3,28,4,4,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
840512,31,2026-03-11 14:00:00+02:00,8,0,1.0,1.0,15.2,15.2,31.98,-1.4,0.0,0.0,0,11.9,196.3,1025.0,20.0,0.0,0.0,0,19.0,6.0,6.0,0.0,8.0,4.0,0.0,0.0,0.0,15.0,15.0,15.0,2.0,2.0,6.0,4.0,0.0,66.0,457.0,3.0,-1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.225396,7.0,0.410116,0.388734,0.0,0.0,0.31613,0.332111,0.857143,3.0,0.281373,30.0,0.900000,2026,3,11,14,14,2,0
840513,31,2026-03-11 15:00:00+02:00,8,0,1.0,1.0,15.4,15.4,29.33,-2.4,0.0,0.0,0,12.2,200.6,1024.0,20.0,0.0,0.0,0,13.0,0.0,0.0,0.0,3.0,3.0,0.0,0.0,0.0,2.0,2.0,2.0,7.0,7.0,7.0,3.0,2.0,50.0,456.0,-6.0,-1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.225396,7.0,0.410116,0.388734,0.0,0.0,0.31613,0.332111,0.857143,3.0,0.281373,30.0,0.900000,2026,3,11,15,15,2,0
840514,31,2026-03-11 16:00:00+02:00,8,0,1.0,1.0,15.1,15.1,29.03,-2.8,0.0,0.0,0,11.5,202.3,1024.0,20.0,0.0,0.0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,32.0,439.0,0.0,-1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.225396,7.0,0.410

In [80]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 840517 entries, 0 to 840516
Data columns (total 72 columns):
 #   Column                     Non-Null Count   Dtype                      
---  ------                     --------------   -----                      
 0   region_id                  840517 non-null  int64                      
 1   time                       840517 non-null  datetime64[us, Europe/Kyiv]
 2   region                     840517 non-null  int64                      
 3   alarm                      840517 non-null  int64                      
 4   has_started                840517 non-null  float64                    
 5   has_ended                  840517 non-null  float64                    
 6   temp                       840517 non-null  float64                    
 7   feelslike                  840517 non-null  float64                    
 8   humidity                   840517 non-null  float64                    
 9   dew                        840517 non-null  floa

In [81]:
cols_to_int = ['messages_count', 'has_threat_sum',
       'nlp_артобстрілу', 'nlp_бпла', 'nlp_відбій', 'nlp_відбій_тривоги',
       'nlp_дніпропетровська', 'nlp_донецька', 'nlp_запорізька',
       'nlp_нікополь', 'nlp_нікополь_нікопольська', 'nlp_нікопольська',
       'nlp_повітряна', 'nlp_повітряна_тривога', 'nlp_тривога', 'nlp_тривоги',
       'nlp_харківська', 'msg_count_last_3h', 'msg_count_last_24h',
       'threat_diff_1h', 'day_of_week', 'is_weekend',
       'text_length', 'isw_cluster_0', 'isw_cluster_1', 'isw_cluster_2',
       'isw_cluster_3', 'isw_cluster_4', 'isw_cluster_5', 'isw_cluster_6',
       'isw_cluster_7', 'isw_cluster_8', 'isw_cluster_9', "news_count_7d", 
        "news_count_30d", "anomaly_count_7d", "anomaly_count_30d",
        "news_velocity_7d", "news_velocity_30d"]

df[cols_to_int] = df[cols_to_int].astype(int)

In [82]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 840517 entries, 0 to 840516
Data columns (total 72 columns):
 #   Column                     Non-Null Count   Dtype                      
---  ------                     --------------   -----                      
 0   region_id                  840517 non-null  int64                      
 1   time                       840517 non-null  datetime64[us, Europe/Kyiv]
 2   region                     840517 non-null  int64                      
 3   alarm                      840517 non-null  int64                      
 4   has_started                840517 non-null  float64                    
 5   has_ended                  840517 non-null  float64                    
 6   temp                       840517 non-null  float64                    
 7   feelslike                  840517 non-null  float64                    
 8   humidity                   840517 non-null  float64                    
 9   dew                        840517 non-null  floa

# Saving result

In [83]:
save_path = Path("data/merged/merged_v7.csv")

In [84]:
df.to_csv(save_path, index=False)